# UPAR 2027 Track 2: fine-tuned EVA-02 Large

Run from top to bottom on Kaggle with **GPU and Internet enabled**. The notebook downloads only the official challenge data, loads the generic `timm/eva02_large_patch14_448.mim_in22k_ft_in22k_in1k` weights (about 1.2 GB, MIT licence, not gated), fine-tunes the whole network on the 40 provided attributes, selects the checkpoint by validation retrieval mADM, and writes `/kaggle/working/upar_eva02_large_submission.zip`.

EVA-02-L/14 was pretrained with masked image modelling on ImageNet-22k and then fine-tuned on ImageNet-22k and ImageNet-1k (about 90.0% ImageNet-1k top-1 at 448 x 448). No pedestrian or PAR data was used for those weights. Here the 448 x 448 position embedding is resampled to **336 x 168** so that pedestrian crops keep their aspect ratio and one epoch stays affordable.

The submission ZIP contains a TorchScript export of the fine-tuned model and needs neither `timm` nor network access at inference. It is about 1.1 GB. The notebook prints full validation mADM, mAP, Rank-1/5/10, and mINP before packaging. Compare those numbers with the current DINOv3 result before uploading.


## 1. Download the official challenge training and validation data

These data cells follow the existing ConvNeXt Tiny Kaggle notebook. They may be skipped if `/kaggle/working/upar_2027/data` is already complete.


In [1]:
%pip -q install gdown tqdm requests timm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import shutil
import subprocess
import zipfile
from pathlib import Path

import gdown
import requests
from tqdm.auto import tqdm

WORK = Path('/kaggle/working')
REPO = WORK / 'UPAR-Challenge-2027'
DATA = WORK / 'upar_2027' / 'data'
DATA.mkdir(parents=True, exist_ok=True)

if not (REPO / 'peta_file_mapping.txt').exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/speckean/UPAR-Challenge-2027.git', str(REPO)
    ], check=True)
for split in ('train', 'val'):
    destination = DATA / 'annotations' / f'{split}.csv'
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPO / 'data' / 'annotations' / f'{split}.csv', destination)
print('Data directory:', DATA)

Cloning into '/kaggle/working/UPAR-Challenge-2027'...


Data directory: /kaggle/working/upar_2027/data


In [3]:
# The annotation paths determine exactly which images must be available.
needed = {'Market1501': set(), 'PA100k': set(), 'PETA': set()}
for split in ('train', 'val'):
    with (DATA / 'annotations' / f'{split}.csv').open(newline='') as handle:
        for row in list(csv.reader(handle))[1:]:
            relative = Path(row[0])
            needed[relative.parts[0]].add(relative)

def missing(source):
    return [path for path in needed[source] if not (DATA / path).is_file()]

def check_zip(path):
    if path.exists() and not zipfile.is_zipfile(path):
        path.unlink()  # Remove an interrupted download before retrying.
    return path.exists()

def require_complete(source):
    absent = missing(source)
    if absent:
        raise RuntimeError(f'{source}: {len(absent)} images missing; examples: {absent[:5]}')
    print(f'{source}: {len(needed[source]):,} annotated images available')

print({source: len(paths) for source, paths in needed.items()})

{'Market1501': 26458, 'PA100k': 88987, 'PETA': 15631}


## Market1501

In [4]:
market_archive = DATA / 'market_1501.zip'
market_dir = DATA / 'Market1501'
if missing('Market1501'):
    if not check_zip(market_archive):
        result = gdown.download(
            'https://drive.google.com/file/d/0B8-rUzbwVRk0c054eEozWG9COHM/view?resourcekey=0-8nyl7K9_x37HlQm34MmrYQ',
            output=str(market_archive), quiet=False, use_cookies=False, fuzzy=True
        )
        if not result or not zipfile.is_zipfile(market_archive):
            raise RuntimeError('Market1501 download failed or is not a ZIP file')
    with zipfile.ZipFile(market_archive) as archive:
        archive.extractall(DATA)
    extracted = DATA / 'Market-1501-v15.09.15'
    if extracted.exists():
        shutil.copytree(extracted, market_dir, dirs_exist_ok=True)
        shutil.rmtree(extracted)
require_complete('Market1501')
market_archive.unlink(missing_ok=True)

Downloading...
From (original): https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM
From (redirected): https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM&confirm=t&uuid=636fe1fc-07f5-4a46-9629-d634c7e95d5b
To: /kaggle/working/upar_2027/data/market_1501.zip
100%|██████████| 153M/153M [00:00<00:00, 230MB/s]


Market1501: 26,458 annotated images available


## PA-100K

In [5]:
pa_dir = DATA / 'PA100k'
pa_dir.mkdir(exist_ok=True)
pa_archive = pa_dir / 'data.zip'
if missing('PA100k'):
    if not check_zip(pa_archive):
        gdown.download_folder(
            'https://drive.google.com/drive/folders/1d_D0Yh7C262gr0ef9EqkvG_M3fqgAWa2?usp=sharing',
            output=str(pa_dir), quiet=False, use_cookies=False
        )
        if not zipfile.is_zipfile(pa_archive):
            raise RuntimeError('PA-100K data.zip was not downloaded')
    with zipfile.ZipFile(pa_archive) as archive:
        archive.extractall(pa_dir)
require_complete('PA100k')
pa_archive.unlink(missing_ok=True)

Retrieving folder contents


Processing file 1dt5nteusNfCVrjAkffLK1sbh4hTLM8tl annotation.zip
Processing file 1Bod3MrbdCRiSvpR7NSOYi-rF3-ushwtj data.zip
Processing file 1zmQkoN2YJgbclT990ihj_DYU29YAidey README.txt


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1dt5nteusNfCVrjAkffLK1sbh4hTLM8tl
To: /kaggle/working/upar_2027/data/PA100k/annotation.zip
100%|██████████| 339k/339k [00:00<00:00, 86.5MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Bod3MrbdCRiSvpR7NSOYi-rF3-ushwtj
From (redirected): https://drive.google.com/uc?id=1Bod3MrbdCRiSvpR7NSOYi-rF3-ushwtj&confirm=t&uuid=97089595-a2eb-4865-bb19-9947b6be3810
To: /kaggle/working/upar_2027/data/PA100k/data.zip
100%|██████████| 451M/451M [00:03<00:00, 138MB/s]
Downloading...
From: https://drive.google.com/uc?id=1zmQkoN2YJgbclT990ihj_DYU29YAidey
To: /kaggle/working/upar_2027/data/PA100k/README.txt
100%|██████████| 794/794 [00:00<00:00, 2.69MB/s]
Download completed


PA100k: 88,987 annotated images available


## PETA

Only the mapped images are extracted from the archive, directly to their challenge filenames.

In [6]:
peta_dir = DATA / 'PETA'
peta_dir.mkdir(exist_ok=True)
peta_archive = peta_dir / 'peta.zip'
if missing('PETA'):
    if not check_zip(peta_archive):
        url = 'https://www.dropbox.com/s/52ylx522hwbdxz6/PETA.zip?dl=1'
        temporary = peta_archive.with_suffix('.zip.part')
        with requests.get(url, stream=True, timeout=(30, 120)) as response:
            response.raise_for_status()
            total = int(response.headers.get('content-length', 0))
            with temporary.open('wb') as output, tqdm(total=total or None, unit='B', unit_scale=True, desc='PETA') as bar:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        output.write(chunk)
                        bar.update(len(chunk))
        temporary.replace(peta_archive)
        if not zipfile.is_zipfile(peta_archive):
            raise RuntimeError('PETA download is not a ZIP file')
    with (REPO / 'peta_file_mapping.txt').open(newline='') as handle:
        mapping = list(csv.reader(handle))
    with zipfile.ZipFile(peta_archive) as archive:
        members = set(archive.namelist())
        for source, destination in tqdm(mapping, desc='Extracting PETA images'):
            target = DATA / destination
            if target.is_file():
                continue
            member = source.removeprefix('PETA/')
            if member not in members:
                raise FileNotFoundError(f'PETA archive is missing {member}')
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as original, target.open('wb') as output:
                shutil.copyfileobj(original, output)
require_complete('PETA')
peta_archive.unlink(missing_ok=True)

PETA:   0%|          | 0.00/236M [00:00<?, ?B/s]

Extracting PETA images:   0%|          | 0/19000 [00:00<?, ?it/s]

PETA: 15,631 annotated images available


In [7]:
for source in needed:
    require_complete(source)
print('Ready:', DATA)
print('Train annotations:', DATA / 'annotations' / 'train.csv')
print('Validation annotations:', DATA / 'annotations' / 'val.csv')

Market1501: 26,458 annotated images available
PA100k: 88,987 annotated images available
PETA: 15,631 annotated images available
Ready: /kaggle/working/upar_2027/data
Train annotations: /kaggle/working/upar_2027/data/annotations/train.csv
Validation annotations: /kaggle/working/upar_2027/data/annotations/val.csv


## 2. Fine-tune EVA-02 Large and package the submission

All 303 M parameters are trained with AdamW, layer-wise learning-rate decay, a short warm-up, and cosine decay. The loss is unweighted binary cross-entropy over the 40 logits. After every epoch the notebook scores a fixed set of 256 validation queries and keeps the checkpoint with the highest mADM in `upar_eva02_large_best.pth`.

Edit the settings at the top of the cell as needed. Gradient checkpointing is on because a batch of 32 does not otherwise fit a 16 GB GPU; it trades roughly a third more training time for much lower memory. If the GPU still runs out of memory, reduce `BATCH_SIZE` to 16. `IMAGE_SIZE = (448, 224)` uses the native patch density but costs roughly twice as much per epoch. Both sides must be multiples of 14.


In [8]:
"""Kaggle training and offline submission for UPAR 2027 Track 2.

EVA-02 Large experiment: an ImageNet-22k pretrained and fine-tuned EVA-02-L/14
is fine-tuned end to end on the 40 challenge attributes. The checkpoint and the
ranking distance are selected by validation retrieval mADM.
"""

import csv
import importlib.util
import json
import math
import os
import random
import shutil
import tempfile
import zipfile
from pathlib import Path

import numpy as np
import timm
import torch
from PIL import Image
from timm.optim import create_optimizer_v2
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode


DATA_ROOT = Path(os.environ.get("UPAR_DATA_ROOT", "/kaggle/working/upar_2027/data"))
OUTPUT_ROOT = Path("/kaggle/working" if Path("/kaggle/working").exists() else ".")
MODEL_ID = "eva02_large_patch14_448.mim_in22k_ft_in22k_in1k"  # ImageNet only; no PAR weights.
USE_PRETRAINED_WEIGHTS = True  # Kaggle training only; no runtime download.
IMAGE_SIZE = (336, 168)  # height, width; both must be multiples of the 14-pixel patch.
EPOCHS = 3
BATCH_SIZE = 32  # Reduce to 16 if the Kaggle GPU still runs out of memory.
GRAD_CHECKPOINTING = True  # Recomputes block activations in the backward pass; needed on a 16 GB GPU.
EVAL_BATCH_SIZE = 128
NUM_WORKERS = 2
LEARNING_RATE = 1e-4  # Head learning rate; earlier blocks are scaled down by LAYER_DECAY.
LAYER_DECAY = 0.75
WEIGHT_DECAY = 0.05
DROP_PATH_RATE = 0.1
WARMUP_STEPS = 500
LOG_EVERY = 200
SELECTION_QUERIES = 256
SEED = 42


def read_annotations(path):
    with path.open(newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader)
        # The released train.csv omits the image-column name in its header.
        names = header if len(header) == 40 else header[1:]
        names = [name.removeprefix("# ").strip() for name in names]
        if len(names) != 40:
            raise ValueError(f"Expected 40 attributes in {path}, got {len(names)}")
        paths, labels = [], []
        for row_number, row in enumerate(reader, 2):
            if len(row) != 41:
                raise ValueError(f"{path}:{row_number}: expected image path and 40 labels")
            paths.append(row[0])
            labels.append([int(value) for value in row[1:]])
    labels = np.asarray(labels, dtype=np.float32)
    if not np.isin(labels, [0, 1]).all():
        raise ValueError(f"{path} contains labels other than 0 and 1")
    return names, paths, labels


class AttributeDataset(Dataset):
    def __init__(self, root, paths, labels, transform):
        self.root, self.paths, self.labels, self.transform = root, paths, labels, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.root / self.paths[index]) as image:
            tensor = self.transform(image.convert("RGB"))
        return tensor, torch.from_numpy(self.labels[index])


def build_transforms(mean, std):
    resize = transforms.Resize(IMAGE_SIZE, interpolation=InterpolationMode.BICUBIC)
    normalize = transforms.Normalize(mean, std)
    # No hue or saturation jitter: 24 of the 40 attributes are clothing colours.
    train_transform = transforms.Compose([
        resize,
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(IMAGE_SIZE, padding=10, padding_mode="edge"),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        normalize,
    ])
    val_transform = transforms.Compose([resize, transforms.ToTensor(), normalize])
    return train_transform, val_transform


def build_optimizer(model):
    optimizer = create_optimizer_v2(model, opt="adamw", lr=LEARNING_RATE,
                                    weight_decay=WEIGHT_DECAY, layer_decay=LAYER_DECAY)
    # timm stores the layer-wise factor as lr_scale; PyTorch schedulers ignore it.
    for group in optimizer.param_groups:
        group["lr"] = LEARNING_RATE * group.get("lr_scale", 1.0)
    return optimizer


@torch.inference_mode()
def predict_logits(model, loader, device):
    model.eval()
    pieces = []
    for images, _ in loader:
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            logits = model(images.to(device, non_blocking=True))
        pieces.append(logits.float().cpu().numpy())
    return np.concatenate(pieces)


def distances_for(logits, queries, method):
    if method == "nll":
        return np.logaddexp(0, logits).sum(axis=1)[None, :] - queries @ logits.T
    if method == "l1":
        probabilities = 1 / (1 + np.exp(-np.clip(logits, -30, 30)))
        return probabilities.sum(axis=1)[None, :] + queries @ (1 - 2 * probabilities).T
    raise ValueError(method)


def retrieval_metrics(logits, labels, queries, method="nll"):
    """Challenge mADM, mAP, Rank-1/5/10, mINP for a validation gallery."""
    labels = labels.astype(np.float32)
    queries = queries.astype(np.float32)
    gallery_sum = labels.sum(axis=1)
    totals = np.zeros(6, dtype=np.float64)
    for start in range(0, len(queries), 16):
        query_batch = queries[start:start + 16]
        distances = distances_for(logits, query_batch, method)
        agreement = 40 - gallery_sum[None, :] - query_batch.sum(axis=1)[:, None] + 2 * query_batch @ labels.T
        for row in range(len(query_batch)):
            order = np.argsort(distances[row], kind="stable")
            ranks = np.flatnonzero(agreement[row, order] == 40) + 1
            if not len(ranks):
                raise ValueError("Validation query has no exact gallery match")
            mean_agreement = agreement[row].mean()
            normalized = np.maximum(0, (agreement[row, order] - mean_agreement) / (40 - mean_agreement))
            prefix = np.cumsum(normalized)
            totals += [np.mean(prefix[ranks - 1] / ranks),
                       np.mean(np.arange(1, len(ranks) + 1) / ranks),
                       ranks[0] <= 1, ranks[0] <= 5, ranks[0] <= 10,
                       len(ranks) / ranks[-1]]
    keys = ("mADM", "mAP", "Rank-1", "Rank-5", "Rank-10", "mINP")
    return dict(zip(keys, (totals / len(queries)).tolist()))


RUN_PY = '''"""Offline EVA-02 Large attribute retrieval for UPAR 2027 Track 2."""
from pathlib import Path
import json

import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

HERE = Path(__file__).resolve().parent
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL = None
CONFIG = None
TRANSFORM = None
BATCH_SIZE = 64


def load_model():
    global MODEL, CONFIG, TRANSFORM
    with (HERE / "assets/config.json").open() as handle:
        CONFIG = json.load(handle)
    TRANSFORM = transforms.Compose([
        transforms.Resize(tuple(CONFIG["image_size"]), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(CONFIG["mean"], CONFIG["std"]),
    ])
    MODEL = torch.jit.load(str(HERE / "assets/model.pt"), map_location=DEVICE).eval()


def rank_gallery(sample):
    if MODEL is None:
        load_model()
    names = sample["attribute_names"]
    indices = [CONFIG["attribute_names"].index(name) for name in names]
    queries = np.asarray(sample["queries"], dtype=np.float32)
    if queries.ndim != 2 or queries.shape[1] != len(indices):
        raise ValueError("Expected a Q x 40 query matrix")
    gallery = sample["gallery"]
    distances = np.empty((len(queries), len(gallery)), dtype=np.float32)
    with torch.inference_mode():
        for start in range(0, len(gallery), BATCH_SIZE):
            batch = gallery[start:start + BATCH_SIZE]
            pixels = []
            for item in batch:
                with Image.open(item["image_path"]) as image:
                    pixels.append(TRANSFORM(image.convert("RGB")))
            with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
                logits = MODEL(torch.stack(pixels).to(DEVICE)).float().cpu().numpy()[:, indices]
            if CONFIG["score_method"] == "nll":
                block = np.logaddexp(0, logits).sum(axis=1)[None, :] - queries @ logits.T
            else:
                probs = 1 / (1 + np.exp(-np.clip(logits, -30, 30)))
                block = probs.sum(axis=1)[None, :] + queries @ (1 - 2 * probs).T
            distances[:, start:start + len(batch)] = block
            if (start // BATCH_SIZE + 1) % 100 == 0:
                print(f"Ranked {start + len(batch):,}/{len(gallery):,} images", flush=True)
    if not np.isfinite(distances).all():
        raise ValueError("Nonfinite distances")
    return {"distances": distances}
'''

METADATA_YAML = '''# Code submissions are executed by the challenges ingestion program, which
# imports run.py and calls rank_gallery(). Nothing in this
# file is used for inference, it exists only so the platform never treats the
# archive as a result submission. Leave it as-is.
command: python3 -c "print('Submission package ready. The ingestion program imports run.py')"
'''


def package_submission(model, names, score_method, mean, std, val_transform, example_path, example_query):
    folder = OUTPUT_ROOT / "upar_eva02_large_submission"
    assets = folder / "assets"
    assets.mkdir(parents=True, exist_ok=True)
    model = model.cpu().eval()
    # The runner has no timm, so the model ships as TorchScript. Plain attention
    # keeps the traced graph loadable by the runner's older PyTorch.
    for module in model.modules():
        if hasattr(module, "fused_attn"):
            module.fused_attn = False
    with Image.open(example_path) as image:
        example = val_transform(image.convert("RGB")).unsqueeze(0)
    with torch.inference_mode():
        traced = torch.jit.trace(model, example, check_trace=False)
        for batch in (example, example.repeat(2, 1, 1, 1)):
            if not torch.allclose(model(batch), traced(batch), atol=1e-4, rtol=1e-4):
                raise ValueError("TorchScript export changed model predictions")
        traced.save(str(assets / "model.pt"))
    (assets / "config.json").write_text(json.dumps({
        "attribute_names": names, "score_method": score_method,
        "image_size": list(IMAGE_SIZE), "mean": list(mean), "std": list(std),
        "model_id": MODEL_ID,
    }, indent=2))
    (folder / "run.py").write_text(RUN_PY)
    (folder / "metadata.yaml").write_text(METADATA_YAML)
    archive_path = OUTPUT_ROOT / "upar_eva02_large_submission.zip"
    expected = {"run.py", "metadata.yaml", "assets/model.pt", "assets/config.json"}
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in sorted(expected):
            archive.write(folder / name, name)
    with zipfile.ZipFile(archive_path) as archive:
        print("ZIP contents:", archive.namelist())
        assert set(archive.namelist()) == expected
        with tempfile.TemporaryDirectory() as temp_dir:
            archive.extractall(temp_dir)
            spec = importlib.util.spec_from_file_location("eva02_submission_check", Path(temp_dir) / "run.py")
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            result = module.rank_gallery({
                "attribute_names": names,
                "queries": [example_query.astype(int).tolist()],
                "gallery": [{"index": 0, "image_path": str(example_path)}],
            })
            distances = np.asarray(result["distances"])
            assert distances.shape == (1, 1) and np.isfinite(distances).all()
            print("Packaged inference smoke test passed")
    print(f"Upload this ZIP to Codabench: {archive_path} ({archive_path.stat().st_size / 1e6:,.0f} MB)")


def main():
    if not (DATA_ROOT / "annotations/train.csv").is_file():
        raise FileNotFoundError(
            f"Missing {DATA_ROOT / 'annotations/train.csv'}. Run the download cells above "
            "in this Kaggle session or attach its saved output and set UPAR_DATA_ROOT."
        )
    if IMAGE_SIZE[0] % 14 or IMAGE_SIZE[1] % 14:
        raise ValueError("IMAGE_SIZE must be a multiple of the 14-pixel patch size")
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.backends.cudnn.benchmark = True

    names, train_paths, train_labels = read_annotations(DATA_ROOT / "annotations/train.csv")
    val_names, val_paths, val_labels = read_annotations(DATA_ROOT / "annotations/val.csv")
    if names != val_names:
        raise ValueError("Train and validation attribute orders differ")
    for relative in train_paths[:10] + val_paths[:10]:
        if not (DATA_ROOT / relative).is_file():
            raise FileNotFoundError(f"Missing image {DATA_ROOT / relative}; finish downloading the images")
    print(f"Loaded {len(train_paths):,} train and {len(val_paths):,} validation images")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # timm resamples the 448 x 448 position embedding to the pedestrian aspect ratio.
    model = timm.create_model(MODEL_ID, pretrained=USE_PRETRAINED_WEIGHTS, num_classes=40,
                              img_size=IMAGE_SIZE, drop_path_rate=DROP_PATH_RATE)
    mean, std = model.pretrained_cfg["mean"], model.pretrained_cfg["std"]
    model.set_grad_checkpointing(GRAD_CHECKPOINTING)
    model.to(device)

    train_transform, val_transform = build_transforms(mean, std)
    train_loader = DataLoader(
        AttributeDataset(DATA_ROOT, train_paths, train_labels, train_transform),
        batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS,
        pin_memory=device.type == "cuda",
    )
    val_loader = DataLoader(
        AttributeDataset(DATA_ROOT, val_paths, val_labels, val_transform),
        batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=device.type == "cuda",
    )

    criterion = nn.BCEWithLogitsLoss()
    optimizer = build_optimizer(model)
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = min(WARMUP_STEPS, total_steps // 10)

    def lr_factor(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")

    queries = np.unique(val_labels, axis=0).astype(np.float32)
    rng = np.random.default_rng(SEED)
    selected = queries[rng.choice(len(queries), min(SELECTION_QUERIES, len(queries)), replace=False)]

    checkpoint_path = OUTPUT_ROOT / "upar_eva02_large_best.pth"
    best_score, best_logits = -1.0, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running, seen = 0.0, 0
        for step, (images, labels) in enumerate(train_loader, 1):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running += loss.item() * len(images)
            seen += len(images)
            if step % LOG_EVERY == 0:
                print(f"Epoch {epoch} step {step:,}/{len(train_loader):,}: BCE={running / seen:.4f}", flush=True)

        logits = predict_logits(model, val_loader, device)
        metrics = retrieval_metrics(logits, val_labels, selected, "nll")
        print(f"Epoch {epoch}/{EPOCHS}: train BCE={running / seen:.4f}; selection={metrics}", flush=True)
        if metrics["mADM"] > best_score:
            best_score, best_logits = metrics["mADM"], logits
            torch.save({
                "state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},
                "attribute_names": names,
                "image_size": IMAGE_SIZE,
                "model_id": MODEL_ID,
                "epoch": epoch,
                "selection_mADM": best_score,
            }, checkpoint_path)
            print(f"Saved {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["state_dict"])
    print(f"Restored epoch {checkpoint['epoch']} with selection mADM={best_score:.4f}")
    candidates = {name: retrieval_metrics(best_logits, val_labels, selected, name)
                  for name in ("nll", "l1")}
    score_method = max(candidates, key=lambda name: candidates[name]["mADM"])
    print("Selection scoring comparison:", candidates)
    print("Chosen score:", score_method)
    print("Full validation metrics:", retrieval_metrics(best_logits, val_labels, queries, score_method))

    del optimizer, scaler
    model.set_grad_checkpointing(False)
    model.cpu()
    torch.cuda.empty_cache()
    package_submission(model, names, score_method, mean, std, val_transform,
                       DATA_ROOT / val_paths[0], val_labels[0])



## 3. Run training, validation, and build the offline submission ZIP

This cell reports the validation retrieval metrics after every epoch, selects NLL or L1 scoring on the fixed query subset, exports TorchScript, and checks the ZIP with a one-image inference call. Only the supplied training annotations are used for learning.


In [9]:
main()

Loaded 97,669 train and 33,407 validation images


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 1 step 200/3,052: BCE=0.4149
Epoch 1 step 400/3,052: BCE=0.3217
Epoch 1 step 600/3,052: BCE=0.2647
Epoch 1 step 800/3,052: BCE=0.2278
Epoch 1 step 1,000/3,052: BCE=0.2027
Epoch 1 step 1,200/3,052: BCE=0.1850
Epoch 1 step 1,400/3,052: BCE=0.1713
Epoch 1 step 1,600/3,052: BCE=0.1605
Epoch 1 step 1,800/3,052: BCE=0.1519
Epoch 1 step 2,000/3,052: BCE=0.1448
Epoch 1 step 2,200/3,052: BCE=0.1388
Epoch 1 step 2,400/3,052: BCE=0.1336
Epoch 1 step 2,600/3,052: BCE=0.1291
Epoch 1 step 2,800/3,052: BCE=0.1251
Epoch 1 step 3,000/3,052: BCE=0.1215
Epoch 1/3: train BCE=0.1207; selection={'mADM': 0.7629530271971651, 'mAP': 0.38545018149988114, 'Rank-1': 0.38671875, 'Rank-5': 0.6484375, 'Rank-10': 0.73828125, 'mINP': 0.2561066502454444}
Saved /kaggle/working/upar_eva02_large_best.pth
Epoch 2 step 200/3,052: BCE=0.0649
Epoch 2 step 400/3,052: BCE=0.0639
Epoch 2 step 600/3,052: BCE=0.0638
Epoch 2 step 800/3,052: BCE=0.0631
Epoch 2 step 1,000/3,052: BCE=0.0625
Epoch 2 step 1,200/3,052: BCE=0.0621
E

/usr/local/lib/python3.13/dist-packages/torch/__init__.py:2253: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not condition:


ZIP contents: ['assets/config.json', 'assets/model.pt', 'metadata.yaml', 'run.py']
Packaged inference smoke test passed
Upload this ZIP to Codabench: /kaggle/working/upar_eva02_large_submission.zip (1,125 MB)


Download `upar_eva02_large_submission.zip` from Kaggle Outputs. Submit it only after comparing validation mADM and runtime with the DINOv3 result.
